#### By: Peyman Shahidi
#### Created: Apr 26, 2026
#### Last Edit: Apr 26, 2026

<br>

Compute pairwise **Euclidean distance** between task embeddings within each O\*NET occupation, using **enriched task descriptions** (Task Title + Occupation Title + DWA Titles + top Skills + top Tools/Technology). Tasks are encoded with a sentence-transformer language model; per-occupation distance tables and top-k closest neighbors are written to `data/computed_objects/taskHandoffCost/`.

> Note: only `Task Title` and `DWA Titles` actually vary across tasks within an occupation, so the occupation-level enrichments (Occupation Title, Skills, Tools) shift absolute distances but contribute little to *within-occupation* relative distances. They are included primarily for disambiguation of polysemous task titles and for cross-occupation comparability.

In [1]:
#Python
import getpass
import os
import numpy as np
import pandas as pd
from collections import defaultdict
import itertools
import random

## formatting number to appear comma separated and with two digits after decimal: e.g, 1000 shown as 1,000.00
pd.set_option('float_format', "{:,.2f}".format)

import matplotlib.pyplot as plt
#%matplotlib inline
#from matplotlib.legend import Legend

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 200)

In [2]:
main_folder_path = ".."
input_data_path = f"{main_folder_path}/data"

# Sample mode: set to a positive int to run on only that many occupations
# (writes to a separate `_sample` folder so the production output stays clean).
# Set to None to process all occupations.
sample_size = None

# Number of nearest neighbors to record per Task ID 1 in the top-closest file
top_k = 2

output_subfolder = 'taskHandoffCost_sample' if sample_size else 'taskHandoffCost'
output_data_path = f'{input_data_path}/computed_objects/{output_subfolder}'
pairwise_output_path = f'{output_data_path}/pairwise'
topk_output_path = f'{output_data_path}/topClosestTasks'
output_plot_path = f"{main_folder_path}/writeup/plots/{output_subfolder}"

In [3]:
# Create directories if they don't exist
for path in [pairwise_output_path, topk_output_path, output_plot_path]:
    if not os.path.exists(path):
        os.makedirs(path)

# 1) Read O*NET Dataset and Build Enriched Task Text

For each task we build a single string combining the **Task Title**, **Occupation Title**, the task's **DWA Titles** (one task may map to multiple DWAs), the occupation's **top-5 Skills** by Importance, and the occupation's **top-10 Tools/Technology** (Hot Technology prioritized). This enriched text is what we later feed to the embedding model.

In [ ]:
# Read O*NET data
ONET = pd.read_csv(f'{input_data_path}/computed_objects/ONET_cleaned_tasks.csv')

# Drop DWA columns to avoid double counting
# (we'll re-attach DWAs aggregated per-task below for enrichment)
ONET = ONET.drop(columns=['DWA ID', 'DWA Title'])

# Remove duplicate rows
rows_before = len(ONET)
print(f"Number of rows before removing duplicates: {rows_before:,}")
ONET = ONET.drop_duplicates().reset_index(drop=True)
rows_after = len(ONET)
print(f"Number of rows after removing duplicates: {rows_after:,}")
print(f"Duplicates removed: {rows_before - rows_after}")

# Keep the columns we need for handoff-cost computation
ONET = ONET[['O*NET-SOC Code', 'Occupation Title', 'Task ID', 'Task Title']].reset_index(drop=True)

# If in sample mode, restrict to the first `sample_size` occupations (deterministic by SOC code)
if sample_size:
    sampled_codes = sorted(ONET['O*NET-SOC Code'].unique())[:sample_size]
    ONET = ONET[ONET['O*NET-SOC Code'].isin(sampled_codes)].reset_index(drop=True)
    print(f"\nSAMPLE MODE: restricted to {sample_size} occupations -> {len(ONET):,} task rows")

# ----- Enrichment data: per-task DWAs, per-occupation top Skills and Tools -----

# (1) Per-task DWA titles (a task may map to multiple DWAs; concatenate them)
tasks_to_dwas = pd.read_csv(f'{input_data_path}/db_27_3_text/Tasks to DWAs.txt', sep='\t')
dwa_reference = pd.read_csv(f'{input_data_path}/db_27_3_text/DWA Reference.txt', sep='\t')
task_dwas = tasks_to_dwas.merge(dwa_reference[['DWA ID', 'DWA Title']], on='DWA ID', how='left')
per_task_dwas = (
    task_dwas.groupby(['O*NET-SOC Code', 'Task ID'])
             .agg(dwa_titles=('DWA Title', lambda x: ' | '.join(x.dropna().unique())))
             .reset_index()
)

# (2) Per-occupation top-5 Skills by Importance (Scale ID = 'IM')
skills_df = pd.read_csv(f'{input_data_path}/db_27_3_text/Skills.txt', sep='\t')
skills_im = skills_df[skills_df['Scale ID'] == 'IM'].copy()
top_skills = (
    skills_im.sort_values(['O*NET-SOC Code', 'Data Value'], ascending=[True, False])
             .groupby('O*NET-SOC Code')
             .head(5)
             .groupby('O*NET-SOC Code')
             .agg(top_skills=('Element Name', lambda x: ', '.join(x)))
             .reset_index()
)

# (3) Per-occupation top-10 Tools / Technology (Hot Technology prioritized, then Tools Used, then other Tech)
tools_used = pd.read_csv(f'{input_data_path}/db_27_3_text/Tools Used.txt', sep='\t')
tech_skills = pd.read_csv(f'{input_data_path}/db_27_3_text/Technology Skills.txt', sep='\t')
hot_tech = tech_skills[tech_skills['Hot Technology'] == 'Y'][['O*NET-SOC Code', 'Commodity Title']]
other_tech = tech_skills[tech_skills['Hot Technology'] != 'Y'][['O*NET-SOC Code', 'Commodity Title']]
tools_combined = pd.concat([
    hot_tech,
    tools_used[['O*NET-SOC Code', 'Commodity Title']],
    other_tech,
], ignore_index=True).drop_duplicates(subset=['O*NET-SOC Code', 'Commodity Title'])
top_tools = (
    tools_combined.groupby('O*NET-SOC Code')
                  .head(10)
                  .groupby('O*NET-SOC Code')
                  .agg(top_tools=('Commodity Title', lambda x: ', '.join(x)))
                  .reset_index()
)

# ----- Merge enrichments and build the enriched text per task -----
ONET = ONET.merge(per_task_dwas, on=['O*NET-SOC Code', 'Task ID'], how='left')
ONET = ONET.merge(top_skills, on='O*NET-SOC Code', how='left')
ONET = ONET.merge(top_tools, on='O*NET-SOC Code', how='left')

ONET['dwa_titles'] = ONET['dwa_titles'].fillna('')
ONET['top_skills'] = ONET['top_skills'].fillna('')
ONET['top_tools'] = ONET['top_tools'].fillna('')

def build_enriched(row):
    parts = [f"Occupation: {row['Occupation Title']}.",
             f"Task: {row['Task Title']}"]
    if row['dwa_titles']:
        parts.append(f"Activity: {row['dwa_titles']}.")
    if row['top_skills']:
        parts.append(f"Top skills: {row['top_skills']}.")
    if row['top_tools']:
        parts.append(f"Tools: {row['top_tools']}.")
    return ' '.join(parts)

ONET['enriched_text'] = ONET.apply(build_enriched, axis=1)

print(f"\nNumber of unique ONET occupations: {ONET['O*NET-SOC Code'].nunique():,}")
print(f"Number of unique ONET tasks: {ONET['Task ID'].nunique():,}")
print(f"\nExample enriched text (first task):")
print(ONET['enriched_text'].iloc[0])
ONET.head()

# 2) Encode Tasks with a Sentence-Transformer

Each task's **enriched text** is mapped into a 768-dimensional vector with `all-mpnet-base-v2`. Embeddings are L2-normalized, and the resulting Euclidean distances are rescaled to $[0, 1]$ by dividing by 2 (the theoretical max distance between unit vectors), keeping the metric directly comparable to cosine-based measures used elsewhere in the project.

In [5]:
import importlib.metadata, subprocess, sys
def pip_install_if_needed(pkg, min_v):
    try:
        v = importlib.metadata.version(pkg)
        if v >= min_v:
            print(f"{pkg} {v} OK")
            return
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg} not found")
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"{pkg}>={min_v}", "-U"])

pip_install_if_needed("sentence-transformers", "3.1.0")

sentence-transformers 5.4.1 OK


In [6]:
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

# Choose embedding model and device
device = "cpu"
language_model = "all-mpnet-base-v2"  # alt: 'all-MiniLM-L6-v2'
embedding_model_path = f"{input_data_path}/sentence-transformers/{language_model}"

# Load model
model = SentenceTransformer(embedding_model_path, device=device)
print("Model OK; embedding dim =", len(model.encode(["test"])[0]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: ../data/sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model OK; embedding dim = 768


In [ ]:
# Encode every unique task once (using its enriched text), then look up vectors per occupation.
unique_tasks = (
    ONET[['Task ID', 'enriched_text']]
    .drop_duplicates(subset='Task ID')
    .reset_index(drop=True)
)
print(f"Encoding {len(unique_tasks):,} unique tasks (enriched text)...")

texts = unique_tasks['enriched_text'].astype(str).tolist()
emb = model.encode(
    texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
)
E = emb.astype(np.float32)

# Defensive: ensure L2-normalized
norms = np.linalg.norm(E, axis=1)
if not np.allclose(norms, 1.0, atol=1e-3):
    E = normalize(E, norm='l2')

# Map Task ID -> embedding row index
taskid_to_idx = dict(zip(unique_tasks['Task ID'].values, np.arange(len(unique_tasks))))
print("Embeddings shape:", E.shape)

# 3) Pairwise Distances Within Each Occupation

For every occupation, compute the upper-triangular pairwise Euclidean distance between its task embeddings (each unordered pair once, with `Task ID 1 < Task ID 2`) and write the long-format table to `pairwise/{Occupation_Title}.csv`.

In [8]:
from scipy.spatial.distance import pdist

occ_groups = ONET.groupby(['O*NET-SOC Code', 'Occupation Title'], sort=False)
n_pairwise_files = 0
n_pairs_total = 0
n_singletons = 0

for (soc_code, occ_title), group in occ_groups:
    # Sort tasks by Task ID up front so pairwise pairs satisfy Task ID 1 < Task ID 2
    group_sorted = group.sort_values('Task ID').reset_index(drop=True)
    task_ids = group_sorted['Task ID'].values
    task_titles = group_sorted['Task Title'].values
    n = len(task_ids)

    if n < 2:
        n_singletons += 1
        continue

    rows_idx = np.array([taskid_to_idx[t] for t in task_ids])
    E_occ = E[rows_idx]

    # Condensed pairwise distances (each unordered pair once), rescaled to [0, 1]
    raw_dists = pdist(E_occ, metric='euclidean')
    dists_condensed = raw_dists / 2.0

    i_idx, j_idx = np.triu_indices(n, k=1)
    pairwise_out = pd.DataFrame({
        'O*NET-SOC Code': soc_code,
        'Occupation Title': occ_title,
        'Task ID 1': task_ids[i_idx],
        'Task Title 1': task_titles[i_idx],
        'Task ID 2': task_ids[j_idx],
        'Task Title 2': task_titles[j_idx],
        'euclidean_distance': dists_condensed,
    })
    pairwise_out = pairwise_out.sort_values(['Task ID 1', 'Task ID 2']).reset_index(drop=True)

    safe_title = occ_title.replace(" ", "_").replace("/", "_")
    pairwise_out.to_csv(f"{pairwise_output_path}/{safe_title}.csv", index=False)
    n_pairwise_files += 1
    n_pairs_total += len(pairwise_out)

print(f"✓ Wrote {n_pairwise_files:,} pairwise files to {pairwise_output_path}")
print(f"✓ Total task pairs across all occupations: {n_pairs_total:,}")
if n_singletons:
    print(f"⚠  Skipped {n_singletons} occupation(s) with fewer than 2 tasks")

✓ Wrote 873 pairwise files to ../data/computed_objects/taskHandoffCost/pairwise
✓ Total task pairs across all occupations: 193,140


# 4) Top-k Closest Tasks (Post-processing)

Read each `pairwise/{Occupation_Title}.csv` produced above and, for every task, keep its `top_k` lowest-distance neighbors. Output goes to `topClosestTasks/{Occupation_Title}.csv` with one row per (focal task, neighbor) pair, ordered by `Task ID 1` then `rank` (1 = closest).

In [9]:
import glob

pw_files = sorted(glob.glob(f"{pairwise_output_path}/*.csv"))
n_topk_files = 0
n_topk_rows = 0

for pw_path in pw_files:
    pw = pd.read_csv(pw_path)
    fname = os.path.basename(pw_path)

    # Pairwise stores each unordered pair once with Task ID 1 < Task ID 2.
    # Duplicate in both directions so each task can appear as the focal Task ID 1.
    dirA = pw.copy()
    dirB = pw.rename(columns={
        'Task ID 1': 'Task ID 2',
        'Task ID 2': 'Task ID 1',
        'Task Title 1': 'Task Title 2',
        'Task Title 2': 'Task Title 1',
    })
    long_df = pd.concat([dirA, dirB], ignore_index=True)

    # For each focal task, keep the top_k smallest distances
    long_df = long_df.sort_values(['Task ID 1', 'euclidean_distance']).reset_index(drop=True)
    long_df['rank'] = long_df.groupby('Task ID 1').cumcount() + 1
    topk_df = long_df[long_df['rank'] <= top_k].copy()
    topk_df = topk_df.sort_values(['Task ID 1', 'rank']).reset_index(drop=True)

    # Reorder columns
    topk_df = topk_df[['O*NET-SOC Code', 'Occupation Title',
                       'Task ID 1', 'Task Title 1',
                       'Task ID 2', 'Task Title 2',
                       'euclidean_distance', 'rank']]

    topk_df.to_csv(f"{topk_output_path}/{fname}", index=False)
    n_topk_files += 1
    n_topk_rows += len(topk_df)

print(f"✓ Wrote {n_topk_files:,} top-{top_k} closest-task files to {topk_output_path}")
print(f"✓ Total rows across all top-{top_k} files: {n_topk_rows:,}")

✓ Wrote 873 top-2 closest-task files to ../data/computed_objects/taskHandoffCost/topClosestTasks
✓ Total rows across all top-2 files: 35,906


In [10]:
# Sanity check: load one occupation back from each subfolder and inspect
sample_files = sorted(os.listdir(pairwise_output_path))[:1]
if sample_files:
    fname = sample_files[0]

    pw = pd.read_csv(f"{pairwise_output_path}/{fname}")
    print(f"[pairwise] {fname}  ({len(pw):,} pairs)")
    print(pw['euclidean_distance'].describe())
    display(pw.head())

    tk = pd.read_csv(f"{topk_output_path}/{fname}")
    print(f"\n[topClosestTasks] {fname}  ({len(tk):,} rows)")
    display(tk.head(2 * top_k))

[pairwise] Accountants_and_Auditors.csv  (325 pairs)
count   325.00
mean      0.52
std       0.06
min       0.35
25%       0.48
50%       0.52
75%       0.57
max       0.66
Name: euclidean_distance, dtype: float64


,O*NET-SOC Code,Occupation Title,Task ID 1,Task Title 1,Task ID 2,Task Title 2,euclidean_distance
0,13-2011.00,Accountants and Auditors,21505,Prepare detailed reports on audit findings.,21506,Report to management about asset utilization a...,0.42
1,13-2011.00,Accountants and Auditors,21505,Prepare detailed reports on audit findings.,21507,Collect and analyze data to detect deficient c...,0.47
2,13-2011.00,Accountants and Auditors,21505,Prepare detailed reports on audit findings.,21508,Inspect account books and accounting systems f...,0.44
3,13-2011.00,Accountants and Auditors,21505,Prepare detailed reports on audit findings.,21509,"Supervise auditing of establishments, and dete...",0.45
4,13-2011.00,Accountants and Auditors,21505,Prepare detailed reports on audit findings.,21510,Confer with company officials about financial ...,0.47



[topClosestTasks] Accountants_and_Auditors.csv  (52 rows)


,O*NET-SOC Code,Occupation Title,Task ID 1,Task Title 1,Task ID 2,Task Title 2,euclidean_distance,rank
0,13-2011.00,Accountants and Auditors,21505,Prepare detailed reports on audit findings.,21506,Report to management about asset utilization a...,0.42,1
1,13-2011.00,Accountants and Auditors,21505,Prepare detailed reports on audit findings.,21508,Inspect account books and accounting systems f...,0.44,2
2,13-2011.00,Accountants and Auditors,21506,Report to management about asset utilization a...,21505,Prepare detailed reports on audit findings.,0.42,1
3,13-2011.00,Accountants and Auditors,21506,Report to management about asset utilization a...,21520,Report to management regarding the finances of...,0.42,2
